In [3]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from transformers import AutoTokenizer
from src.embed.chunk import chunk_ids, split_sentences

In [2]:
q = pd.read_parquet("../data/processed/speeches.parquet")
tok = AutoTokenizer.from_pretrained("tohoku-nlp/bert-base-japanese-v3")

smp = q.sample(3000, random_state=0).copy()
smp["n_tok"] = [len(tok(t)["input_ids"]) for t in smp["text"]]
smp["n_char"] = smp["text"].str.len()

print(smp["n_tok"].describe(percentiles=[.5, .75, .9, .95, .99]).round(0))
print("\n512トークンを超える発言の割合:", round((smp["n_tok"] > 512).mean(), 3))
print("512を超えた部分が、全トークンに占める割合:",
      round((smp["n_tok"] - 512).clip(lower=0).sum() / smp["n_tok"].sum(), 3))
print("1トークンあたりの文字数:", round(smp["n_char"].sum() / smp["n_tok"].sum(), 2))
print("\n党ごとの中央値:")
print(smp.groupby("party")["n_tok"].median().sort_values())

c:\Users\taiyo\anaconda3\envs\survey_on_party_systems\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\taiyo\.cache\huggingface\hub\models--tohoku-nlp--bert-base-japanese-v3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] Token indices sequence length is longer than the specified maximum 

count    3000.0
mean      318.0
std       301.0
min         5.0
50%       250.0
75%       415.0
90%       626.0
95%       805.0
99%      1411.0
max      5302.0
Name: n_tok, dtype: float64

512トークンを超える発言の割合: 0.17
512を超えた部分が、全トークンに占める割合: 0.153
1トークンあたりの文字数: 1.51

党ごとの中央値:
party
共産     198.0
社民     201.0
立憲     205.0
その他    228.0
れいわ    232.5
維新     264.0
国民     271.0
中道     283.0
公明     286.0
自民     315.0
みらい    327.0
参政     344.0
保守     347.0
Name: n_tok, dtype: float64


In [4]:
q = pd.read_parquet("../data/processed/speeches.parquet")
tok = AutoTokenizer.from_pretrained("tohoku-nlp/bert-base-japanese-v3")

ex = "防衛費について伺います。増額には反対です。大臣の見解は?"
print(split_sentences(ex))
print([tok.decode(c) for c in chunk_ids(ex, tok, max_tokens=8)])

smp = q.sample(3000, random_state=0)
chunks = [chunk_ids(t, tok) for t in smp["text"]]
n_chunks = np.array([len(c) for c in chunks])
lens = np.array([len(x) for c in chunks for x in c])

print(
    "\nかたまりごとの数(発言ごと):", 
    pd.Series(n_chunks).describe(percentiles=[.5, .9, .99]).round(1).to_dict()
)
print(
    "1件あたり平均:", n_chunks.mean().round(2), 
    " / かたまりが0の発言:", (n_chunks == 0).sum()
)
print("かたまりの長さの最大:", lens.max(), "(510以下なら合格)")
print("全体のかたまり数の見積もり:", int(n_chunks.mean() * len(q)))


['防衛費について伺います。', '増額には反対です。', '大臣の見解は?']
['防衛 費 に つい て 伺い ます', '。', '増額 に は 反対 です 。', '大臣 の 見解 は ?']

かたまりごとの数(発言ごと): {'count': 3000.0, 'mean': 1.2, 'std': 0.6, 'min': 1.0, '50%': 1.0, '90%': 2.0, '99%': 3.0, 'max': 11.0}
1件あたり平均: 1.22  / かたまりが0の発言: 0
かたまりの長さの最大: 510 (510以下なら合格)
全体のかたまり数の見積もり: 107310
